# Music Annotation Project V7.7.1: Page010 Forensics and Interpretable Safety Vetoes

**Runtime:** CPU. Select **Runtime > Run all**.

This notebook is a read-only forensic successor to V7.7. It does not rerun reconstruction, refit the V7.7 structural model, change labels, alter source images, or authorise automatic correction.

It answers three questions:

1. Why is Page010 associated with high-confidence structural false-safe cases?
2. How well is structural-safety probability calibrated for each held-out page?
3. Can a simple, interpretable veto prevent false-safe selections without destroying useful coverage?

## Guardrails

- Reads the latest complete V7.7 run, unless `V77_RUN_OVERRIDE` is set.
- Requires V7.7's component dataset and out-of-fold predictions.
- Uses only V7.7 predictor-allowlist variables for veto discovery.
- Never uses raw page coordinates as veto features.
- Never uses clean-truth or outcome variables as veto inputs.
- Page010 is treated as a forensic target, not a permissible production feature.
- Candidate vetoes are retrospective findings, not production policy.
- A zero-observed-failure rule is reported as unavailable when no such policy exists.
- Independent score/source-group validation remains mandatory.

The output is written to a new timestamped `structural_risk_forensics_v7_7_1` directory.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q pandas numpy matplotlib scipy scikit-learn


In [ ]:
import json, math, itertools, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from datetime import datetime, timezone
from scipy.stats import ks_2samp, mannwhitneyu
from sklearn.metrics import brier_score_loss, roc_auc_score, average_precision_score

ROOT=Path('/content/drive/MyDrive/MusicAnnotationProject')
V77_ROOT=ROOT/'structural_risk_calibration_v7_7'
V77_RUN_OVERRIDE=None
OUTPUT_ROOT=ROOT/'structural_risk_forensics_v7_7_1'
TARGET_PAGE='Page010'
RANDOM_SEED=42
MIN_SELECTED=10
MIN_PAGES_SELECTED=2
MAX_SINGLE_VETOES_TO_COMBINE=80
PROBABILITY_THRESHOLDS=np.unique(np.r_[np.linspace(0.50,0.99,50),[0.995,0.999,1.0]])
QUANTILES=[.01,.025,.05,.10,.15,.20,.25,.30,.40,.50,.60,.70,.75,.80,.85,.90,.95,.975,.99]

def complete_run(p):
    req=[p/'v7_7_manifest.json',p/'tables'/'asymmetric_component_dataset.csv',p/'tables'/'out_of_fold_asymmetric_predictions.csv',p/'tables'/'predictor_allowlist.csv']
    return all(x.exists() for x in req)
def newest(root):
    candidates=[]
    if root.exists():
        candidates=[p for p in root.rglob('*') if p.is_dir() and complete_run(p)]
    return max(candidates,key=lambda p:p.stat().st_mtime) if candidates else None
V77_RUN=Path(V77_RUN_OVERRIDE) if V77_RUN_OVERRIDE else newest(V77_ROOT)
if V77_RUN is None or not complete_run(V77_RUN):
    raise RuntimeError('No complete V7.7 run found. Set V77_RUN_OVERRIDE to a directory containing the required V7.7 outputs.')
RUN_NAME=datetime.now().strftime('%Y%m%d_%H%M%S');OUT=OUTPUT_ROOT/RUN_NAME
TABLES=OUT/'tables';CHARTS=OUT/'charts';CONTACTS=OUT/'contact_sheets'
for d in [OUT,TABLES,CHARTS,CONTACTS]:d.mkdir(parents=True,exist_ok=True)
print('V7.7 source:',V77_RUN);print('V7.7.1 output:',OUT)


## Load and validate the frozen V7.7 evidence


In [ ]:
dataset=pd.read_csv(V77_RUN/'tables'/'asymmetric_component_dataset.csv')
pred=pd.read_csv(V77_RUN/'tables'/'out_of_fold_asymmetric_predictions.csv')
allow=pd.read_csv(V77_RUN/'tables'/'predictor_allowlist.csv')
manifest=json.loads((V77_RUN/'v7_7_manifest.json').read_text())
keys=['page','component']
required_pred={'page','component','structural_safe','prob_structural_safe'}
if not required_pred.issubset(pred.columns):raise RuntimeError(f'Prediction file missing: {required_pred-set(pred.columns)}')
df=dataset.merge(pred[[c for c in pred.columns if c not in dataset.columns or c in keys]],on=keys,how='inner',validate='one_to_one')
df['structural_safe']=df.structural_safe.astype(str).str.lower().map({'true':True,'false':False}).fillna(df.structural_safe.astype(bool))
df['prob_structural_safe']=pd.to_numeric(df.prob_structural_safe,errors='coerce')
predictors=[c for c in allow.predictor.astype(str) if c in df.columns]
forbidden_exact={'x','y','x_cache','y_cache','page','component'}
forbidden_patterns=['truth','clean','structural','cosmetic','outcome','failure','ink_recall','ink_f1','edge_f1','component_mae','background_mae','removed_print','residual_annotation']
eligible=[]
for c in predictors:
    if c in forbidden_exact or any(p in c.lower() for p in forbidden_patterns):continue
    if pd.api.types.is_numeric_dtype(df[c]) or pd.to_numeric(df[c],errors='coerce').notna().any():eligible.append(c)
predictors=eligible
if not predictors:raise RuntimeError('No eligible V7.7 predictors remain after the forensic firewall.')
if TARGET_PAGE not in set(df.page.astype(str)):raise RuntimeError(f'{TARGET_PAGE} is absent from the V7.7 dataset.')
assert not {'x','y'}.intersection(predictors)
assert not any(any(p in c.lower() for p in forbidden_patterns) for c in predictors)
print('Components:',len(df),'pages:',df.page.nunique(),'eligible predictors:',len(predictors))
print('Target page components:',int((df.page==TARGET_PAGE).sum()))


## Reconstruct the V7.7 threshold policy correctly


In [ ]:
def wilson_upper(errors,n,z=1.96):
    if n==0:return np.nan
    p=errors/n;den=1+z*z/n;centre=p+z*z/(2*n);radius=z*math.sqrt(p*(1-p)/n+z*z/(4*n*n));return (centre+radius)/den
policy=[]
p=df.prob_structural_safe.to_numpy(float);safe=df.structural_safe.to_numpy(bool);valid=np.isfinite(p)
for threshold in PROBABILITY_THRESHOLDS:
    selected=valid&(p>=threshold);n=int(selected.sum())
    if not n:continue
    failures=int(np.sum(selected&~safe))
    policy.append({'probability_threshold':float(threshold),'selected_components':n,'selected_fraction':n/len(df),'selected_pages':int(df.loc[selected,'page'].nunique()),'structural_failures':failures,'observed_false_safe_rate':failures/n,'false_safe_rate_wilson95_upper':wilson_upper(failures,n),'structural_safe_recall':float(np.sum(selected&safe)/max(safe.sum(),1))})
base_policy=pd.DataFrame(policy).sort_values('probability_threshold')
zero=base_policy[(base_policy.structural_failures==0)&(base_policy.selected_components>=MIN_SELECTED)&(base_policy.selected_pages>=MIN_PAGES_SELECTED)]
base_policy.to_csv(TABLES/'baseline_threshold_policy.csv',index=False)
if zero.empty:
    BASE_THRESHOLD=None
    print('NO ZERO-OBSERVED-FAILURE BASELINE POLICY exists under the minimum coverage constraints.')
else:
    BASE_THRESHOLD=float(zero.sort_values(['selected_components','probability_threshold'],ascending=[False,False]).iloc[0].probability_threshold)
    print('Largest zero-observed-failure baseline threshold:',BASE_THRESHOLD)
# For veto comparisons retain a practical high-score grid, even if none is zero-error.


## Calibration and outcome concentration by page


In [ ]:
def safe_auc(y,p):
    return roc_auc_score(y,p) if len(np.unique(y))==2 else np.nan
def safe_ap(y,p):
    return average_precision_score(y,p) if np.sum(y)>0 else np.nan
page_rows=[]
for page,g in df.groupby('page'):
    valid=g.prob_structural_safe.notna();y=g.loc[valid,'structural_safe'].astype(int);q=g.loc[valid,'prob_structural_safe']
    page_rows.append({'page':page,'components':len(g),'structural_safe':int(g.structural_safe.sum()),'structural_unsafe':int((~g.structural_safe).sum()),'observed_safe_fraction':float(g.structural_safe.mean()),'mean_predicted_safe':float(q.mean()),'median_predicted_safe':float(q.median()),'p90_predicted_safe':float(q.quantile(.90)),'p95_predicted_safe':float(q.quantile(.95)),'p99_predicted_safe':float(q.quantile(.99)),'calibration_error_mean':float(q.mean()-y.mean()),'absolute_calibration_error_mean':float(abs(q.mean()-y.mean())),'brier':brier_score_loss(y,q) if len(y) else np.nan,'roc_auc':safe_auc(y,q),'average_precision':safe_ap(y,q),'maximum_probability':float(q.max()),'unsafe_at_probability_1':int(((q>=1-1e-12)&(y==0)).sum())})
page_df=pd.DataFrame(page_rows).sort_values(['absolute_calibration_error_mean','brier'],ascending=False)
page_df.to_csv(TABLES/'calibration_by_page.csv',index=False)
# Per-page threshold behaviour
rows=[]
for threshold in PROBABILITY_THRESHOLDS:
    for page,g in df.groupby('page'):
        selected=g.prob_structural_safe>=threshold;n=int(selected.sum());fail=int((selected&~g.structural_safe).sum())
        rows.append({'page':page,'probability_threshold':float(threshold),'selected_components':n,'structural_failures':fail,'observed_false_safe_rate':fail/n if n else np.nan,'safe_selected':int((selected&g.structural_safe).sum())})
page_threshold=pd.DataFrame(rows);page_threshold.to_csv(TABLES/'threshold_policy_by_page.csv',index=False)
print(page_df.head(15).to_string(index=False))


In [ ]:
# Page calibration overview
q=page_df.sort_values('calibration_error_mean')
plt.figure(figsize=(14,max(7,.25*len(q))));colours=np.where(q.page==TARGET_PAGE,'crimson','steelblue');plt.barh(q.page,q.calibration_error_mean,color=colours);plt.axvline(0,color='black',linewidth=1);plt.xlabel('Mean predicted safe probability minus observed safe fraction');plt.title('V7.7.1 calibration error by held-out page');plt.tight_layout();plt.savefig(CHARTS/'calibration_error_by_page.png',dpi=180);plt.close()
# Predicted versus observed
plt.figure(figsize=(8,7));plt.scatter(page_df.mean_predicted_safe,page_df.observed_safe_fraction,s=np.maximum(20,page_df.components*2),alpha=.65);target=page_df[page_df.page==TARGET_PAGE]
if len(target):plt.scatter(target.mean_predicted_safe,target.observed_safe_fraction,s=170,color='crimson',edgecolor='black',label=TARGET_PAGE);plt.legend()
plt.plot([0,1],[0,1],'k--');plt.xlabel('Mean predicted structural-safe probability');plt.ylabel('Observed structural-safe fraction');plt.title('Page-level structural-safety calibration');plt.tight_layout();plt.savefig(CHARTS/'page_predicted_vs_observed.png',dpi=180);plt.close()


## Page010 predictor-distribution forensics


In [ ]:
# Compare Page010 with all other pages and with high-score true-safe controls.
records=[]
target=df[df.page==TARGET_PAGE]
other=df[df.page!=TARGET_PAGE]
for c in predictors:
    a=pd.to_numeric(target[c],errors='coerce').dropna();b=pd.to_numeric(other[c],errors='coerce').dropna()
    if len(a)<2 or len(b)<2:continue
    pooled=pd.concat([a,b]);sd=float(pooled.std(ddof=1));effect=(float(a.mean()-b.mean())/sd) if sd>0 else 0.0
    try:ks=ks_2samp(a,b)
    except Exception:ks=type('R',(),{'statistic':np.nan,'pvalue':np.nan})()
    try:mw=mannwhitneyu(a,b,alternative='two-sided')
    except Exception:mw=type('R',(),{'statistic':np.nan,'pvalue':np.nan})()
    records.append({'predictor':c,'page010_n':len(a),'other_n':len(b),'page010_mean':float(a.mean()),'other_mean':float(b.mean()),'page010_median':float(a.median()),'other_median':float(b.median()),'standardised_mean_difference':effect,'absolute_standardised_mean_difference':abs(effect),'ks_statistic':float(ks.statistic),'ks_pvalue':float(ks.pvalue),'mannwhitney_pvalue':float(mw.pvalue),'page010_percentile_of_global_median':float((pooled<=a.median()).mean())})
shift_df=pd.DataFrame(records).sort_values(['absolute_standardised_mean_difference','ks_statistic'],ascending=False)
shift_df.to_csv(TABLES/'page010_predictor_shift.csv',index=False)
print(shift_df.head(20).to_string(index=False))


In [ ]:
q=shift_df.head(20).sort_values('standardised_mean_difference');plt.figure(figsize=(10,8));plt.barh(q.predictor,q.standardised_mean_difference,color=np.where(q.standardised_mean_difference>=0,'darkorange','slateblue'));plt.axvline(0,color='black',linewidth=1);plt.xlabel('Standardised Page010 mean minus other-page mean');plt.title('Largest Page010 predictor shifts');plt.tight_layout();plt.savefig(CHARTS/'page010_predictor_shifts.png',dpi=180);plt.close()


## Explain high-confidence Page010 false-safe cases


In [ ]:
# Comparator groups use the same high-score stratum whenever possible.
HIGH_SCORE=0.95
df['forensic_group']=np.select([
    (df.page==TARGET_PAGE)&(~df.structural_safe)&(df.prob_structural_safe>=HIGH_SCORE),
    (df.page==TARGET_PAGE)&(df.structural_safe)&(df.prob_structural_safe>=HIGH_SCORE),
    (df.page!=TARGET_PAGE)&(df.structural_safe)&(df.prob_structural_safe>=HIGH_SCORE),
    (df.page!=TARGET_PAGE)&(~df.structural_safe)&(df.prob_structural_safe>=HIGH_SCORE),
],['page010_false_safe','page010_true_safe_high','other_true_safe_high','other_false_safe_high'],default='other')
comparisons=[]
focus=df[df.forensic_group=='page010_false_safe']
for comparison_name,control_name in [('vs_page010_true_safe','page010_true_safe_high'),('vs_other_true_safe','other_true_safe_high')]:
    control=df[df.forensic_group==control_name]
    for c in predictors:
        a=pd.to_numeric(focus[c],errors='coerce').dropna();b=pd.to_numeric(control[c],errors='coerce').dropna()
        if len(a)<1 or len(b)<2:continue
        pooled=pd.concat([a,b]);sd=float(pooled.std(ddof=1));effect=(float(a.mean()-b.mean())/sd) if sd>0 else 0.0
        comparisons.append({'comparison':comparison_name,'predictor':c,'false_safe_n':len(a),'control_n':len(b),'false_safe_mean':float(a.mean()),'control_mean':float(b.mean()),'standardised_mean_difference':effect,'absolute_standardised_mean_difference':abs(effect),'false_safe_min':float(a.min()),'false_safe_max':float(a.max()),'control_q05':float(b.quantile(.05)),'control_q95':float(b.quantile(.95))})
explain_df=pd.DataFrame(comparisons).sort_values(['comparison','absolute_standardised_mean_difference'],ascending=[True,False])
explain_df.to_csv(TABLES/'page010_false_safe_feature_contrasts.csv',index=False)
df[df.forensic_group.str.contains('high|false_safe')].to_csv(TABLES/'high_score_forensic_components.csv',index=False)
print('Page010 high-score false-safe components:',len(focus));display(explain_df.groupby('comparison').head(15))


## Exhaustive single-feature veto search


In [ ]:
# A veto is a simple condition that rejects an otherwise selected high-score component.
# Candidate cut-points come only from observed predictor quantiles. Page identity is never a veto.
def evaluate_policy(base_selected,veto,feature,operator,cutoff,kind='single'):
    accepted=base_selected&~veto;n=int(accepted.sum());fail=int((accepted&~df.structural_safe).sum());safe_selected=int((accepted&df.structural_safe).sum())
    base_n=int(base_selected.sum());base_fail=int((base_selected&~df.structural_safe).sum());base_safe=int((base_selected&df.structural_safe).sum())
    rejected=base_selected&veto
    return {'kind':kind,'probability_threshold':float(threshold),'feature':feature,'operator':operator,'cutoff':float(cutoff) if np.isscalar(cutoff) and not isinstance(cutoff,str) else cutoff,'base_selected':base_n,'base_failures':base_fail,'accepted_components':n,'accepted_fraction_all':n/len(df),'accepted_fraction_of_base':n/base_n if base_n else np.nan,'accepted_pages':int(df.loc[accepted,'page'].nunique()),'structural_failures':fail,'observed_false_safe_rate':fail/n if n else np.nan,'false_safe_rate_wilson95_upper':wilson_upper(fail,n),'safe_components_retained':safe_selected,'safe_retention_of_base':safe_selected/base_safe if base_safe else np.nan,'false_safe_prevented':base_fail-fail,'safe_components_rejected':int((rejected&df.structural_safe).sum()),'page010_failures_remaining':int((accepted&(df.page==TARGET_PAGE)&~df.structural_safe).sum()),'page010_components_retained':int((accepted&(df.page==TARGET_PAGE)).sum())}

single=[]
for threshold in PROBABILITY_THRESHOLDS:
    base=df.prob_structural_safe>=threshold
    if base.sum()<MIN_SELECTED:continue
    for c in predictors:
        values=pd.to_numeric(df[c],errors='coerce');finite=values[np.isfinite(values)]
        if finite.nunique()<3:continue
        cuts=np.unique(finite.quantile(QUANTILES).to_numpy())
        for cut in cuts:
            for op,veto in [('>',values>cut),('<',values<cut)]:
                row=evaluate_policy(base,veto.fillna(False).to_numpy(),c,op,cut,'single')
                if row['accepted_components']>=MIN_SELECTED and row['accepted_pages']>=MIN_PAGES_SELECTED:single.append(row)
single_df=pd.DataFrame(single)
single_df.to_csv(TABLES/'single_feature_veto_exhaustive.csv',index=False)
print('Single veto policies tested:',len(single_df))


## Two-veto combinations


In [ ]:
# Combine the best diverse single vetoes at each score threshold. Ranking favours prevented failures, retained safe cases and page breadth.
pairs=[]
for threshold,g in single_df.groupby('probability_threshold'):
    candidates=g.sort_values(['false_safe_prevented','safe_components_rejected','accepted_pages'],ascending=[False,True,False]).drop_duplicates(['feature','operator','cutoff']).head(MAX_SINGLE_VETOES_TO_COMBINE)
    base=(df.prob_structural_safe>=threshold).to_numpy()
    specs=list(candidates[['feature','operator','cutoff']].itertuples(index=False,name=None))
    for a,b in itertools.combinations(specs,2):
        if a[0]==b[0]:continue
        def mask(spec):
            feature,op,cut=spec;v=pd.to_numeric(df[feature],errors='coerce');return ((v>cut) if op=='>' else (v<cut)).fillna(False).to_numpy()
        veto=mask(a)|mask(b);accepted=base&~veto;n=int(accepted.sum())
        if n<MIN_SELECTED or df.loc[accepted,'page'].nunique()<MIN_PAGES_SELECTED:continue
        fail=int((accepted&~df.structural_safe.to_numpy()).sum());base_fail=int((base&~df.structural_safe.to_numpy()).sum());safe_selected=int((accepted&df.structural_safe.to_numpy()).sum());base_safe=int((base&df.structural_safe.to_numpy()).sum())
        rejected=base&veto
        pairs.append({'kind':'pair','probability_threshold':float(threshold),'feature_1':a[0],'operator_1':a[1],'cutoff_1':a[2],'feature_2':b[0],'operator_2':b[1],'cutoff_2':b[2],'base_selected':int(base.sum()),'base_failures':base_fail,'accepted_components':n,'accepted_fraction_all':n/len(df),'accepted_fraction_of_base':n/base.sum(),'accepted_pages':int(df.loc[accepted,'page'].nunique()),'structural_failures':fail,'observed_false_safe_rate':fail/n,'false_safe_rate_wilson95_upper':wilson_upper(fail,n),'safe_components_retained':safe_selected,'safe_retention_of_base':safe_selected/base_safe if base_safe else np.nan,'false_safe_prevented':base_fail-fail,'safe_components_rejected':int((rejected&df.structural_safe.to_numpy()).sum()),'page010_failures_remaining':int((accepted&(df.page.to_numpy()==TARGET_PAGE)&~df.structural_safe.to_numpy()).sum()),'page010_components_retained':int((accepted&(df.page.to_numpy()==TARGET_PAGE)).sum())})
pair_df=pd.DataFrame(pairs);pair_df.to_csv(TABLES/'two_feature_veto_exhaustive.csv',index=False)
print('Two-veto policies tested:',len(pair_df))


## Rank vetoes by structural protection and coverage cost


In [ ]:
def pareto(frame):
    if frame.empty:return frame.copy()
    f=frame.copy();keep=[]
    for i,r in f.iterrows():
        dominated=((f.structural_failures<=r.structural_failures)&(f.safe_components_rejected<=r.safe_components_rejected)&(f.accepted_components>=r.accepted_components)&((f.structural_failures<r.structural_failures)|(f.safe_components_rejected<r.safe_components_rejected)|(f.accepted_components>r.accepted_components))).any()
        keep.append(not dominated)
    return f.loc[keep]
all_policies=pd.concat([single_df,pair_df],ignore_index=True,sort=False)
all_policies['zero_observed_failure']=all_policies.structural_failures==0
all_policies['efficiency_false_safe_prevented_per_safe_rejected']=all_policies.false_safe_prevented/(1+all_policies.safe_components_rejected)
front=pareto(all_policies).sort_values(['structural_failures','safe_components_rejected','accepted_components'],ascending=[True,True,False])
front.to_csv(TABLES/'veto_pareto_front.csv',index=False)
zero=all_policies[(all_policies.structural_failures==0)&(all_policies.accepted_components>=MIN_SELECTED)&(all_policies.accepted_pages>=MIN_PAGES_SELECTED)].sort_values(['accepted_components','safe_components_rejected'],ascending=[False,True])
zero.to_csv(TABLES/'zero_observed_failure_veto_candidates.csv',index=False)
best=all_policies.sort_values(['structural_failures','false_safe_rate_wilson95_upper','safe_components_rejected','accepted_components'],ascending=[True,True,True,False]).head(100)
best.to_csv(TABLES/'best_veto_candidates.csv',index=False)
print('Zero-observed-failure veto candidates:',len(zero));display(best.head(20))


In [ ]:
# Coverage-cost chart.
plot_df=all_policies.copy()
plt.figure(figsize=(10,7));sc=plt.scatter(plot_df.accepted_components,plot_df.false_safe_rate_wilson95_upper,c=plot_df.safe_components_rejected,s=np.maximum(12,25+8*plot_df.false_safe_prevented),alpha=.35,cmap='viridis');plt.colorbar(sc,label='Structurally safe base selections rejected');plt.xlabel('Accepted components after veto');plt.ylabel('95% upper bound on false-safe rate');plt.title('Interpretable veto coverage versus structural-risk uncertainty');plt.tight_layout();plt.savefig(CHARTS/'veto_coverage_vs_risk.png',dpi=180);plt.close()
# Pareto front
if not front.empty:
    plt.figure(figsize=(10,7));plt.scatter(front.accepted_components,front.structural_failures,c=front.safe_components_rejected,cmap='plasma',s=60);plt.colorbar(label='Safe components rejected');plt.xlabel('Accepted components');plt.ylabel('Observed structural failures');plt.title('Pareto front: coverage, failures and safe-case cost');plt.tight_layout();plt.savefig(CHARTS/'veto_pareto_front.png',dpi=180);plt.close()


## Page010-specific overfitting check


In [ ]:
# A candidate is suspicious if its benefit disappears after excluding Page010.
checks=[]
for idx,r in best.head(100).iterrows():
    base=(df.prob_structural_safe>=r.probability_threshold).to_numpy()
    def apply(feature,op,cut):
        if pd.isna(feature):return np.zeros(len(df),bool)
        v=pd.to_numeric(df[feature],errors='coerce');return ((v>cut) if op=='>' else (v<cut)).fillna(False).to_numpy()
    if r.kind=='single':veto=apply(r.feature,r.operator,r.cutoff)
    else:veto=apply(r.feature_1,r.operator_1,r.cutoff_1)|apply(r.feature_2,r.operator_2,r.cutoff_2)
    for scope,scope_mask in [('all',np.ones(len(df),bool)),('excluding_page010',(df.page.to_numpy()!=TARGET_PAGE)),('page010_only',(df.page.to_numpy()==TARGET_PAGE))]:
        accepted=base&~veto&scope_mask;base_scope=base&scope_mask
        checks.append({'candidate_index':int(idx),'kind':r.kind,'scope':scope,'base_selected':int(base_scope.sum()),'accepted':int(accepted.sum()),'base_failures':int((base_scope&~df.structural_safe.to_numpy()).sum()),'remaining_failures':int((accepted&~df.structural_safe.to_numpy()).sum()),'safe_rejected':int((base_scope&veto&df.structural_safe.to_numpy()).sum())})
checks_df=pd.DataFrame(checks);checks_df.to_csv(TABLES/'top_veto_scope_stability.csv',index=False)


## Export manifest and executive summary


In [ ]:
page010_summary=page_df[page_df.page==TARGET_PAGE].iloc[0].to_dict()
recommended=None
if not zero.empty:recommended=zero.iloc[0].to_dict()
manifest_out={'created_utc':datetime.now(timezone.utc).isoformat(),'source_v7_7_run':str(V77_RUN),'read_only_inputs':True,'target_page':TARGET_PAGE,'components':int(len(df)),'pages':int(df.page.nunique()),'eligible_predictors':predictors,'raw_coordinates_excluded':True,'truth_and_outcome_predictors_excluded':True,'page_identity_used_as_veto':False,'single_veto_policies_tested':int(len(single_df)),'two_veto_policies_tested':int(len(pair_df)),'zero_observed_failure_baseline_exists':bool(BASE_THRESHOLD is not None),'zero_observed_failure_veto_candidates':int(len(zero)),'recommended_retrospective_candidate':recommended,'candidate_is_production_authorisation':False,'independent_score_source_group_validation_required':True,'page010_summary':page010_summary}
(OUT/'v7_7_1_manifest.json').write_text(json.dumps(manifest_out,indent=2,default=str))
lines=['V7.7.1 PAGE010 FORENSICS AND INTERPRETABLE SAFETY VETOES','',f'Source V7.7 run: {V77_RUN}',f'Components: {len(df)} across {df.page.nunique()} pages',f'Eligible predictors: {len(predictors)}',f'Page010 components: {int((df.page==TARGET_PAGE).sum())}',f'Page010 observed safe fraction: {page010_summary["observed_safe_fraction"]:.4f}',f'Page010 mean predicted safe probability: {page010_summary["mean_predicted_safe"]:.4f}',f'Page010 unsafe components at probability 1: {int(page010_summary["unsafe_at_probability_1"])}',f'Single vetoes tested: {len(single_df)}',f'Two-veto combinations tested: {len(pair_df)}',f'Zero-observed-failure baseline exists: {BASE_THRESHOLD is not None}',f'Zero-observed-failure veto candidates: {len(zero)}','']
if recommended:
    lines+=['Largest retrospective zero-observed-failure candidate:',json.dumps(recommended,indent=2,default=str),'']
else:
    lines+=['No veto candidate met the stated zero-observed-failure and minimum-coverage constraints.','']
lines+=['IMPORTANT: These are retrospective forensic findings. Page identity is never a veto. Independent score/source-group validation is required before any automatic correction policy.']
(OUT/'executive_summary.txt').write_text(chr(10).join(lines));print(chr(10).join(lines));print('V7.7.1 COMPLETE:',OUT)
